
# Equipment Failure & Predictive Maintenance
## Oil & Gas Industry — Applied Machine Learning

This notebook presents a complete machine learning workflow for predictive maintenance in the oil and gas industry using synthetic sensor data.

The project covers:

- Exploratory Data Analysis (EDA)
- Data Preprocessing
- Feature Engineering
- KMeans Anomaly Detection
- Random Forest Classification
- Model Comparison and Business Interpretation

---


## Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


## Load Dataset

In [ ]:

df = pd.read_csv("equipment_failure_dataset.csv")

print("Dataset Shape:", df.shape)
df.head(10)


## Check Data Types and Missing Values

In [ ]:

print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())


## Summary Statistics

In [ ]:

df.describe()


## Histogram Distribution

In [ ]:

num_cols = df.select_dtypes(include=np.number).columns

df[num_cols].hist(figsize=(15,12))
plt.tight_layout()
plt.show()


## Boxplots for Outlier Detection

In [ ]:

for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot - {col}")
    plt.show()


## Correlation Heatmap

In [ ]:

plt.figure(figsize=(12,8))

corr = df[num_cols].corr()

sns.heatmap(corr, annot=True, cmap="coolwarm")

plt.title("Correlation Heatmap")
plt.show()


## Failure Distribution

In [ ]:

sns.countplot(x='failure_label', data=df)

plt.title("Failure vs Normal Distribution")
plt.show()

failure_percentage = (
    df['failure_label'].mean() * 100
)

print(f"Failure Percentage: {failure_percentage:.2f}%")


## Feature Engineering

In [ ]:

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month

df = df.sort_values(
    ['equipment_id', 'timestamp']
)

for col in ['vibration_mms', 'temperature_c']:

    df[f'{col}_rolling_mean'] = (
        df.groupby('equipment_id')[col]
        .transform(
            lambda x: x.rolling(
                5,
                min_periods=1
            ).mean()
        )
    )

    df[f'{col}_rolling_std'] = (
        df.groupby('equipment_id')[col]
        .transform(
            lambda x: x.rolling(
                5,
                min_periods=1
            ).std()
        )
    )

df.fillna(0, inplace=True)

df.head()


## Train-Test Split

In [ ]:

y = df['failure_label']

X = df.drop(
    columns=[
        'failure_label',
        'timestamp',
        'equipment_id'
    ]
)

X = pd.get_dummies(
    X,
    columns=['equipment_type'],
    drop_first=True
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)


## Preprocessing Pipeline

In [ ]:

numeric_features = X_train.select_dtypes(
    include=np.number
).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            'num',
            Pipeline([
                (
                    'imputer',
                    SimpleImputer(
                        strategy='median'
                    )
                ),
                (
                    'scaler',
                    StandardScaler()
                )
            ]),
            numeric_features
        )
    ],
    remainder='passthrough'
)

X_train_processed = (
    preprocessor.fit_transform(X_train)
)

X_test_processed = (
    preprocessor.transform(X_test)
)

print(X_train_processed.shape)


## Elbow Method for KMeans

In [ ]:

inertia = []

for k in range(2,11):

    km = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    km.fit(X_train_processed)

    inertia.append(km.inertia_)

plt.plot(
    range(2,11),
    inertia,
    marker='o'
)

plt.xlabel("Number of Clusters")
plt.ylabel("Inertia")
plt.title("Elbow Method")

plt.show()


## KMeans Anomaly Detection

In [ ]:

km = KMeans(
    n_clusters=3,
    random_state=42,
    n_init=10
)

km.fit(X_train_processed)

train_distances = np.min(
    km.transform(X_train_processed),
    axis=1
)

test_distances = np.min(
    km.transform(X_test_processed),
    axis=1
)

threshold = np.percentile(
    train_distances,
    95
)

anomaly_flags = (
    test_distances > threshold
).astype(int)

print(
    "Precision:",
    precision_score(
        y_test,
        anomaly_flags
    )
)

print(
    "Recall:",
    recall_score(
        y_test,
        anomaly_flags
    )
)

print(
    "F1 Score:",
    f1_score(
        y_test,
        anomaly_flags
    )
)


## PCA Cluster Visualisation

In [ ]:

pca = PCA(n_components=2)

X_pca = pca.fit_transform(
    X_test_processed
)

plt.figure(figsize=(8,6))

plt.scatter(
    X_pca[:,0],
    X_pca[:,1],
    c=anomaly_flags,
    alpha=0.7
)

plt.title(
    "KMeans Cluster Visualisation"
)

plt.xlabel("PCA Component 1")
plt.ylabel("PCA Component 2")

plt.show()


## Random Forest Classification

In [ ]:

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(
    X_train_processed,
    y_train
)

y_pred = rf.predict(
    X_test_processed
)

print(
    classification_report(
        y_test,
        y_pred
    )
)


## Confusion Matrix

In [ ]:

cm = confusion_matrix(
    y_test,
    y_pred
)

plt.figure(figsize=(6,4))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.show()


## Feature Importance

In [ ]:

feature_names = X_train.columns

importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': rf.feature_importances_
})

importance_df = (
    importance_df
    .sort_values(
        'Importance',
        ascending=False
    )
)

top10 = importance_df.head(10)

plt.figure(figsize=(10,6))

sns.barplot(
    data=top10,
    x='Importance',
    y='Feature'
)

plt.title(
    "Top 10 Feature Importances"
)

plt.show()

top10


## Preprocessing Summary Table

In [ ]:

summary = pd.DataFrame({
    "Metric":[
        "Original Shape",
        "Final Shape",
        "Missing Values Handled",
        "New Features Created",
        "Scaling Method"
    ],
    "Value":[
        str(df.shape),
        str(X.shape),
        "Median Imputation",
        "Hour, Day, Month, Rolling Mean, Rolling Std",
        "StandardScaler"
    ]
})

summary


## Model Comparison

In [ ]:

comparison = pd.DataFrame({
    "Criterion":[
        "Requires Labels",
        "Precision",
        "Recall",
        "F1-Score",
        "Interpretability"
    ],
    "KMeans":[
        "No",
        precision_score(y_test, anomaly_flags),
        recall_score(y_test, anomaly_flags),
        f1_score(y_test, anomaly_flags),
        "Moderate"
    ],
    "Random Forest":[
        "Yes",
        "From Classification Report",
        "From Classification Report",
        "From Classification Report",
        "High"
    ]
})

comparison


## Business Interpretation

In [ ]:

print("""
Business Interpretation:

1. If no historical labels exist, KMeans is useful because
   it can identify unusual sensor behaviour without labelled data.

2. Based on feature importance and EDA findings, the most
   important sensors may include:
   - Vibration
   - Temperature
   - Pressure

3. If equipment enters the top 5% anomaly threshold,
   maintenance teams should inspect the equipment immediately
   to prevent catastrophic failure.

4. False Positives:
   - Increased maintenance cost
   - Unnecessary downtime

5. False Negatives:
   - Equipment damage
   - Safety risk
   - Production losses
""")
